# ComStock → OSM Batch Generator
**Pipeline:** Download building ZIP from OpenEI → Edit `home.xml` + `schedules.csv` → Run OpenStudio simulation → Collect `in.osm`

Based on NREL ComStock 2025 Release 3 / ResStock workflow (Joe Robertson).

---
**Prerequisites:**
- OpenStudio 3.9.0 installed (Windows: `C:/openstudio-3.9.0`, Mac: `/usr/local/openstudio-3.9.0`)
- ResStock repo checked out at tag `2025_Release_1`
- Python packages: `requests`, `lxml`  → `pip install requests lxml`

In [1]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "requests", "lxml", "-q"])
print("Dependencies ready.")

Dependencies ready.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
# ── Cell 2: CONFIGURATION — edit these before running ─────────────────────────
import platform
from pathlib import Path

# ── ResStock repo root (must be checked out at tag 2025_Release_1) ────────────
RESSTOCK_REPO = Path("/path/to/resstock")           # ← EDIT THIS

# ── OpenStudio executable (auto-detected by OS) ───────────────────────────────
if platform.system() == "Windows":
    OPENSTUDIO_EXE = Path("C:/openstudio-3.9.0/bin/openstudio.exe")
else:  # macOS
    OPENSTUDIO_EXE = Path("/usr/local/openstudio-3.9.0/bin/openstudio")

# ── Output directory for harvested .osm files ─────────────────────────────────
OSM_OUTPUT_DIR = Path("./osm_outputs")
OSM_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Scratch directory for downloads / simulation working dirs ─────────────────
WORK_DIR = Path("./comstock_work")
WORK_DIR.mkdir(parents=True, exist_ok=True)

# ── Base S3 URL pattern (OpenEI) ──────────────────────────────────────────────
BASE_URL = (
    "https://oedi-data-lake.s3.amazonaws.com/"
    "nrel-pds-building-stock/end-use-load-profiles-for-us-building-stock/"
    "2025/comstock_amy2018_release_3/building_energy_models/"
    "upgrade={upgrade:02d}/bldg{bldg_id:07d}-up{upgrade:02d}.zip"
)

# ── Buildings × upgrades to process ──────────────────────────────────────────
# Format: list of (building_id, upgrade_id) tuples
BATCH = [
    (1, 0),
    (2, 0),
    (3, 0),
    # add more as needed ...
]

# ── Weather file mapping: FIPS county code → EPW filename ─────────────────────
# The home.xml EPWFilePath will be auto-read from the XML;
# override here only if you want to force a specific EPW.
EPW_OVERRIDE = None   # e.g. Path("/path/to/G0100590.epw") or None to auto-detect

print(f"OS detected      : {platform.system()}")
print(f"OpenStudio exe   : {OPENSTUDIO_EXE}")
print(f"ResStock repo    : {RESSTOCK_REPO}")
print(f"Buildings queued : {len(BATCH)}")

OS detected      : Darwin
OpenStudio exe   : /usr/local/openstudio-3.9.0/bin/openstudio
ResStock repo    : /path/to/resstock
Buildings queued : 3


In [3]:
# ── Cell 3: Helper utilities ───────────────────────────────────────────────────
import os
import shutil
import zipfile
import logging
import requests
import subprocess
import csv
from lxml import etree
from datetime import datetime

# ── Logging setup ─────────────────────────────────────────────────────────────
log_path = WORK_DIR / f"batch_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(log_path),
        logging.StreamHandler(sys.stdout),
    ]
)
log = logging.getLogger("comstock_osm")


def bldg_tag(bldg_id: int, upgrade: int) -> str:
    """Canonical folder/file name, e.g. bldg0000001-up00"""
    return f"bldg{bldg_id:07d}-up{upgrade:02d}"


# ── Step 1: Download ───────────────────────────────────────────────────────────
def download_zip(bldg_id: int, upgrade: int, dest_dir: Path) -> Path:
    """Download building ZIP from OpenEI S3. Returns local zip path."""
    url = BASE_URL.format(bldg_id=bldg_id, upgrade=upgrade)
    zip_path = dest_dir / f"{bldg_tag(bldg_id, upgrade)}.zip"
    if zip_path.exists():
        log.info(f"  ZIP already exists, skipping download: {zip_path.name}")
        return zip_path
    log.info(f"  Downloading: {url}")
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(zip_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
    log.info(f"  Downloaded → {zip_path.name} ({zip_path.stat().st_size / 1e6:.1f} MB)")
    return zip_path


# ── Step 2: Unzip ──────────────────────────────────────────────────────────────
def unzip_to_repo(zip_path: Path, bldg_id: int, upgrade: int) -> Path:
    """Unzip into RESSTOCK_REPO top-level directory. Returns bldg folder path."""
    tag = bldg_tag(bldg_id, upgrade)
    bldg_dir = RESSTOCK_REPO / tag
    if bldg_dir.exists():
        log.info(f"  Bldg dir already exists, re-using: {bldg_dir}")
        return bldg_dir
    log.info(f"  Unzipping {zip_path.name} → {RESSTOCK_REPO}")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(RESSTOCK_REPO)
    if not bldg_dir.exists():
        raise FileNotFoundError(f"Expected folder not found after unzip: {bldg_dir}")
    return bldg_dir


# ── Step 3: Edit home.xml ──────────────────────────────────────────────────────
SCHEDULES_CSV_COLUMNS_TO_DROP = [
    "Vacancy",
    "Power Outage",
    "No Space Heating",
    "No Space Cooling",
]

def edit_home_xml(bldg_dir: Path) -> None:
    """
    Patch home.xml:
      - All <ScheduleFilePath> → relative Cambium CSV paths (unchanged, just validated)
      - <SchedulesFilePath>   → schedules.csv
      - <EPWFilePath>         → ../weather/<filename>.epw  (or EPW_OVERRIDE)
    """
    xml_path = bldg_dir / "home.xml"
    if not xml_path.exists():
        raise FileNotFoundError(f"home.xml not found in {bldg_dir}")

    parser = etree.XMLParser(remove_blank_text=False)
    tree = etree.parse(str(xml_path), parser)
    root = tree.getroot()
    ns = root.nsmap.get(None, "")

    def tag(name):
        return f"{{{ns}}}{name}" if ns else name

    changes = 0

    # ── SchedulesFilePath ─────────────────────────────────────────────────────
    for el in root.iter(tag("SchedulesFilePath")):
        el.text = "schedules.csv"
        changes += 1

    # ── EPWFilePath ───────────────────────────────────────────────────────────
    for el in root.iter(tag("EPWFilePath")):
        if EPW_OVERRIDE:
            el.text = str(EPW_OVERRIDE)
        else:
            # Keep existing path but normalise to forward slashes & relative prefix
            existing = el.text or ""
            if existing and not existing.startswith("../weather/"):
                epw_file = Path(existing).name
                el.text = f"../weather/{epw_file}"
        changes += 1

    # ── ScheduleFilePath (emissions Cambium CSVs) ─────────────────────────────
    # These already contain the correct relative path in the ZIP;
    # we just normalise to forward slashes for cross-platform safety.
    for el in root.iter(tag("ScheduleFilePath")):
        if el.text:
            el.text = el.text.replace("\\", "/")
        changes += 1

    tree.write(str(xml_path), pretty_print=True, xml_declaration=True, encoding="UTF-8")
    log.info(f"  home.xml patched ({changes} elements updated)")


# ── Step 4: Copy & clean schedules.csv ────────────────────────────────────────
def prepare_schedules(bldg_dir: Path) -> None:
    """Copy in.schedules.csv → schedules.csv, dropping unwanted columns."""
    src = bldg_dir / "in.schedules.csv"
    dst = bldg_dir / "schedules.csv"
    if not src.exists():
        raise FileNotFoundError(f"in.schedules.csv not found: {src}")

    with open(src, newline="", encoding="utf-8") as fin:
        reader = csv.DictReader(fin)
        fieldnames = [f for f in reader.fieldnames if f not in SCHEDULES_CSV_COLUMNS_TO_DROP]
        rows = [{k: v for k, v in row.items() if k in fieldnames} for row in reader]

    with open(dst, "w", newline="", encoding="utf-8") as fout:
        writer = csv.DictWriter(fout, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    dropped = set(reader.fieldnames) - set(fieldnames)
    log.info(f"  schedules.csv written (dropped columns: {', '.join(dropped) if dropped else 'none'})")


# ── Step 5: Run OpenStudio simulation ─────────────────────────────────────────
def run_simulation(bldg_dir: Path) -> bool:
    """
    Runs:
      openstudio resources/hpxml-measures/workflow/run_simulation.rb \
          -x <bldg_dir>/home.xml -d
    Returns True on success.
    """
    run_script = RESSTOCK_REPO / "resources" / "hpxml-measures" / "workflow" / "run_simulation.rb"
    cmd = [
        str(OPENSTUDIO_EXE),
        str(run_script),
        "-x", str(bldg_dir / "home.xml"),
        "-d",
    ]
    log.info(f"  Running simulation: {' '.join(cmd)}")
    result = subprocess.run(
        cmd,
        cwd=str(RESSTOCK_REPO),
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        log.error(f"  Simulation FAILED (exit {result.returncode})")
        log.error(f"  STDERR: {result.stderr[-2000:]}")  # last 2k chars
        return False
    log.info("  Simulation completed successfully")
    return True


# ── Step 6: Harvest in.osm ────────────────────────────────────────────────────
def harvest_osm(bldg_dir: Path, bldg_id: int, upgrade: int) -> Path | None:
    """Copy in.osm from run/ folder to OSM_OUTPUT_DIR. Returns destination path."""
    osm_src = bldg_dir / "run" / "in.osm"
    if not osm_src.exists():
        log.error(f"  in.osm not found at: {osm_src}")
        return None
    osm_dst = OSM_OUTPUT_DIR / f"{bldg_tag(bldg_id, upgrade)}.osm"
    shutil.copy2(osm_src, osm_dst)
    log.info(f"  OSM saved → {osm_dst}")
    return osm_dst


print("Helpers loaded ✓")

Helpers loaded ✓


In [4]:
# ── Cell 4: Validation checks before running ──────────────────────────────────
errors = []

if not RESSTOCK_REPO.is_dir():
    errors.append(f"ResStock repo not found: {RESSTOCK_REPO}")

if not OPENSTUDIO_EXE.exists():
    errors.append(f"OpenStudio exe not found: {OPENSTUDIO_EXE}")

run_rb = RESSTOCK_REPO / "resources" / "hpxml-measures" / "workflow" / "run_simulation.rb"
if RESSTOCK_REPO.is_dir() and not run_rb.exists():
    errors.append(f"run_simulation.rb not found — is repo at tag 2025_Release_1? Expected: {run_rb}")

if errors:
    for e in errors:
        print(f"❌ {e}")
    raise SystemExit("Fix configuration errors above before proceeding.")
else:
    print("✅ All pre-flight checks passed.")
    print(f"   ResStock repo : {RESSTOCK_REPO}")
    print(f"   OpenStudio    : {OPENSTUDIO_EXE}")
    print(f"   Batch size    : {len(BATCH)} buildings")

❌ ResStock repo not found: /path/to/resstock
❌ OpenStudio exe not found: /usr/local/openstudio-3.9.0/bin/openstudio


SystemExit: Fix configuration errors above before proceeding.

/Users/tcharan/.pyenv/versions/3.13.2/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3675: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
# ── Cell 5: BATCH RUNNER ──────────────────────────────────────────────────────
# Processes each (building_id, upgrade) tuple through the full pipeline.
# Results are tracked in a summary dict for the report cell below.

results = {}  # tag → {"status": str, "osm": Path|None, "error": str|None}

for bldg_id, upgrade in BATCH:
    tag = bldg_tag(bldg_id, upgrade)
    log.info(f"{'═'*60}")
    log.info(f"Processing: {tag}")
    log.info(f"{'═'*60}")

    results[tag] = {"status": "pending", "osm": None, "error": None}
    bldg_work = WORK_DIR / tag
    bldg_work.mkdir(parents=True, exist_ok=True)

    try:
        # 1. Download
        zip_path = download_zip(bldg_id, upgrade, bldg_work)

        # 2. Unzip to repo
        bldg_dir = unzip_to_repo(zip_path, bldg_id, upgrade)

        # 3. Edit home.xml
        edit_home_xml(bldg_dir)

        # 4. Prepare schedules
        prepare_schedules(bldg_dir)

        # 5. Run simulation
        success = run_simulation(bldg_dir)
        if not success:
            results[tag]["status"] = "sim_failed"
            continue

        # 6. Harvest OSM
        osm_path = harvest_osm(bldg_dir, bldg_id, upgrade)
        if osm_path:
            results[tag]["status"] = "success"
            results[tag]["osm"] = osm_path
        else:
            results[tag]["status"] = "osm_missing"

    except Exception as exc:
        log.exception(f"  Unhandled error for {tag}: {exc}")
        results[tag]["status"] = "error"
        results[tag]["error"] = str(exc)

log.info("Batch complete.")

In [ ]:
# ── Cell 6: Results summary ────────────────────────────────────────────────────
from IPython.display import display
import pandas as pd

rows = []
for tag, info in results.items():
    rows.append({
        "Building": tag,
        "Status": info["status"],
        "OSM Path": str(info["osm"]) if info["osm"] else "",
        "Error": info["error"] or "",
    })

df = pd.DataFrame(rows)
total = len(df)
succeeded = (df["Status"] == "success").sum()
failed = total - succeeded

print(f"\n{'─'*50}")
print(f" Batch complete: {succeeded}/{total} succeeded, {failed} failed")
print(f" Log file: {log_path}")
print(f"{'─'*50}\n")

def color_status(val):
    colors = {
        "success": "background-color: #d4edda; color: #155724",
        "sim_failed": "background-color: #f8d7da; color: #721c24",
        "osm_missing": "background-color: #fff3cd; color: #856404",
        "error": "background-color: #f8d7da; color: #721c24",
        "pending": "background-color: #d1ecf1; color: #0c5460",
    }
    return colors.get(val, "")

display(df.style.applymap(color_status, subset=["Status"]))

In [ ]:
# ── Cell 7: Optional cleanup — remove bldg folders from repo after harvest ─────
# Uncomment to delete extracted building folders to save disk space.
# Only folders with a successfully harvested OSM are removed.

CLEANUP = False   # ← set to True to enable

if CLEANUP:
    for bldg_id, upgrade in BATCH:
        tag = bldg_tag(bldg_id, upgrade)
        if results.get(tag, {}).get("status") == "success":
            bldg_dir = RESSTOCK_REPO / tag
            if bldg_dir.exists():
                shutil.rmtree(bldg_dir)
                log.info(f"Cleaned up: {bldg_dir}")
    print("Cleanup complete.")
else:
    print("Cleanup skipped (CLEANUP=False).")